# AFML SLM — Track D on Kaggle: SFT → preference pairs → DPO → evaluation

**Run with *Save Version → Save & Run All*.** The whole notebook is about 1 hour
on a T4, so one batch run does every phase.

**Before you start (right-hand panel):**
1. *Session options* → Accelerator → **GPU T4 x2** (or P100) and Internet → **On**
2. *Add Input* → the notebook(s) holding the checkpoints: `gqa_base/final.pt`,
   an early `gqa_base/step_*.pt`, and (to skip SFT) a `gqa_sft/final.pt`.
   Cell 3 finds them automatically.

| Phase | Switch | Output (Alpaca path) | Output with `CHAIN_DATASET = "oasst1"` |
|---|---|---|---|
| SFT | `RUN_SFT` | `checkpoints/gqa_sft/final.pt` | `checkpoints/gqa_sft_oasst1/final.pt` |
| pairs | `RUN_PAIRS` | `data/preference/pairs.jsonl` | (reuses the committed pairs) |
| DPO | `RUN_DPO` | `checkpoints/gqa_dpo/final.pt` | `checkpoints/gqa_dpo_oasst1/final.pt` |
| eval | `RUN_EVAL` | `results/eval/` (metrics + samples) | `results/eval_oasst1/` |
| benchmark | `RUN_BENCH` | `results/benchmark_enable_gqa/` (re-run after the SDPA GQA fix) | same |

Set `SMOKE = True` first for a 5-minute test on a few examples.

### Human-written SFT data: the `CHAIN_DATASET` switch

Alpaca is model-generated, and the TA advised against synthetic data (decision O6).
Set `CHAIN_DATASET = "oasst1"` (or `"dolly"`) in the settings cell to run
**SFT -> DPO -> evaluate** on that dataset. It turns `RUN_SFT`, `RUN_DPO` and `RUN_EVAL`
on, and `RUN_PAIRS` off (DPO reuses the committed `data/preference/pairs.jsonl`, so the
two chains differ only in the SFT data). Every output gets a `_oasst1` suffix, so the
Alpaca results are never overwritten. The exact commands it runs:

```
python scripts/sft.py --preset gqa --checkpoint checkpoints/gqa_base/final.pt --dataset oasst1 --run-name gqa_sft_oasst1
python scripts/dpo.py --sft-checkpoint checkpoints/gqa_sft_oasst1/final.pt --run-name gqa_dpo_oasst1
python scripts/evaluate.py --base checkpoints/gqa_base/final.pt --sft checkpoints/gqa_sft_oasst1/final.pt --dpo checkpoints/gqa_dpo_oasst1/final.pt --out results/eval_oasst1
```

Inputs: only `gqa_base/final.pt`. Step-by-step instructions: `docs/RUNBOOK.md`.

## 1. Settings

In [ ]:
# ---------------------------------------------------------------- switches
SMOKE       = False      # True: tiny run to check everything works (~5 min)
RUN_SFT     = False     # False: use a teammate's SFT checkpoint (gqa_sft/final.pt) from Add Input
RUN_PAIRS   = True
RUN_DPO     = True
RUN_EVAL    = True
RUN_BENCH   = False      # re-run MHA vs GQA benchmark (random weights, ~15 min)

SFT_DATASET = "alpaca"   # only used if RUN_SFT = True: "alpaca" | "dolly" | "oasst1"
WEAK_STEP   = 2000       # early checkpoint for rejected samples: step_<WEAK_STEP>.pt
PREV_OUTPUT = None       # to reuse a previous run's outputs: "/kaggle/input/<this-notebook-output>/AFML_miniproject"

# ============ HUMAN-WRITTEN SFT CHAIN: SFT -> DPO -> evaluate ===============
# None     -> the Alpaca path, controlled by the switches above (unchanged).
# "oasst1" -> SFT on OpenAssistant oasst1, then DPO and evaluation, written to
#             checkpoints/gqa_sft_oasst1, checkpoints/gqa_dpo_oasst1,
#             results/gqa_sft_oasst1, results/gqa_dpo_oasst1, results/eval_oasst1.
# "dolly"  -> the same with Dolly-15k (*_dolly).
CHAIN_DATASET = None
# ============================================================================

BRANCH      = "shikshith/final-prep"   # has the separate-run support; "main" once merged
# ---------------------------------------------------------------------------

if CHAIN_DATASET:
    RUN_SFT, RUN_PAIRS, RUN_DPO, RUN_EVAL = True, False, True, True
    SFT_DATASET = CHAIN_DATASET
# A non-Alpaca SFT run gets its own names all the way down, so nothing from the
# Alpaca chain (gqa_sft, gqa_dpo, results/eval) is overwritten.
SUFFIX   = f"_{SFT_DATASET}" if (RUN_SFT and SFT_DATASET != "alpaca") else ""
SFT_RUN  = f"gqa_sft{SUFFIX}"
DPO_RUN  = f"gqa_dpo{SUFFIX}"
EVAL_OUT = f"results/eval{SUFFIX}"
print(f"SFT -> checkpoints/{SFT_RUN}   DPO -> checkpoints/{DPO_RUN}   eval -> {EVAL_OUT}")

import os, subprocess, sys, shutil
from pathlib import Path

REPO = "https://github.com/Shikshith05/AFML_miniproject.git"
ROOT = Path("/kaggle/working/AFML_miniproject")
os.chdir("/kaggle/working")  # step out of ROOT before deleting it (re-runs)
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("at", Path.cwd(), subprocess.run(["git", "log", "--oneline", "-1"],
                                       capture_output=True, text=True).stdout)
if SUFFIX and "--ref-cache" not in Path("scripts/dpo.py").read_text():
    raise SystemExit(f"branch {BRANCH!r} has no separate-run support (scripts/sft.py would "
                     "overwrite gqa_sft): set BRANCH = 'shikshith/final-prep' or a branch that has it")

In [ ]:
!pip install -q tokenizers datasets ftfy
import torch
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU, then rerun"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Checkpoints

Searches everything under `/kaggle/input` for `gqa_base/final.pt` and the
weak `step_*.pt`, and copies them to `checkpoints/gqa_base/`. With
`PREV_OUTPUT` set, also brings back SFT/DPO checkpoints and pairs from an
earlier run so finished phases can be skipped.

In [ ]:
dest = Path("checkpoints/gqa_base"); dest.mkdir(parents=True, exist_ok=True)
found = list(Path("/kaggle/input").rglob("*.pt"))
base = [p for p in found if p.name == "final.pt" and "gqa_base" in str(p)]
steps = sorted((p for p in found if p.name.startswith("step_") and "gqa_base" in str(p)),
               key=lambda p: int(p.stem.split("_")[1]))
print("found:", *[str(p) for p in found], sep="\n  ")
if not base:
    raise SystemExit("no gqa_base/final.pt under /kaggle/input -- Add Input -> the checkpoint dataset")
shutil.copy(base[0], dest / "final.pt")

# weak checkpoint: the requested step, else the earliest available
# names are zero-padded (step_002000.pt), so compare the number, not the text
weak = next((p for p in steps if int(p.stem.split("_")[1]) == WEAK_STEP), steps[0] if steps else None)
if weak is not None:
    shutil.copy(weak, dest / weak.name)
    WEAK_CKPT = str(dest / weak.name)
else:
    WEAK_CKPT = None
    print("WARNING: no early step_*.pt found -- pairs cannot be built (see notes, fallback)")
print("base:", dest / "final.pt", "\nweak:", WEAK_CKPT)

# existing SFT checkpoint (Mythri's Alpaca run, notebook60e6ad04ff) attached as input
sft = [p for p in found if p.name == "final.pt" and "gqa_sft" in str(p)]
if sft:
    Path("checkpoints/gqa_sft").mkdir(parents=True, exist_ok=True)
    shutil.copy(sft[0], "checkpoints/gqa_sft/final.pt")
    print("sft :", sft[0], "-> checkpoints/gqa_sft/final.pt")
    # bring its training log along so it ends up in the results zip
    for m in Path("/kaggle/input").rglob("results/gqa_sft/metrics.jsonl"):
        Path("results/gqa_sft").mkdir(parents=True, exist_ok=True)
        shutil.copy(m, "results/gqa_sft/metrics.jsonl")
        print("sft log:", m)
        break
elif not RUN_SFT and (RUN_DPO or RUN_EVAL):
    raise SystemExit("RUN_SFT is False but no gqa_sft/final.pt under /kaggle/input -- "
                     "Add Input -> the SFT notebook (notebook60e6ad04ff)")

if PREV_OUTPUT:
    prev = Path(PREV_OUTPUT)
    for sub in [f"checkpoints/{SFT_RUN}", f"checkpoints/{DPO_RUN}", "data/preference"]:
        if (prev / sub).exists():
            shutil.copytree(prev / sub, sub, dirs_exist_ok=True)
            print("restored", sub)
    for f in prev.glob("data/ref_logprobs_*.npz"):
        shutil.copy(f, "data/")

## 3. SFT

Skipped when `RUN_SFT = False` (a teammate's SFT checkpoint is used instead).
Writes to `checkpoints/{SFT_RUN}` (`gqa_sft`, or `gqa_sft_oasst1` with `CHAIN_DATASET`).
Otherwise fine-tunes `gqa_base/final.pt` on instruction data. Prints a masking check
first (prompt tokens ignored, response tokens learned), then val loss before
and after each epoch.

In [ ]:
if RUN_SFT:
    cmd = (f"python scripts/sft.py --preset gqa --checkpoint checkpoints/gqa_base/final.pt"
           f" --dataset {SFT_DATASET} --run-name {SFT_RUN}")
    if SMOKE:
        cmd += " --limit 2000 --epochs 1"
    print(cmd, flush=True)
    get_ipython().system(cmd)
else:
    print("RUN_SFT is False -- skipping")

## 4. Preference pairs

chosen = real ending of a held-out story, rejected = the weak checkpoint's
ending from the same opening. Check the printed example and the mean
lengths: a big gap between chosen and rejected lengths is a confound to
report.

In [ ]:
if RUN_PAIRS:
    if WEAK_CKPT is None:
        raise SystemExit("no weak checkpoint -- see section 2")
    cmd = f"python scripts/build_preference_data.py --weak-checkpoint {WEAK_CKPT}"
    if SMOKE:
        cmd += " --n-pairs 200"
    print(cmd, flush=True)
    get_ipython().system(cmd)
else:
    print("RUN_PAIRS is False -- skipping")

## 5. DPO

Starts from the SFT model, which is also the frozen reference. Before
training the test loss must be 0.693 (= log 2) and reward accuracy 0; after
training, reward accuracy should be well above 0.5.
The reference log-prob cache is fingerprinted with the SFT model's weights, and a
non-Alpaca run uses its own `data/ref_logprobs_{DPO_RUN}_*.npz`, so the Alpaca cache is never reused.

In [ ]:
if RUN_DPO:
    cmd = f"python scripts/dpo.py --sft-checkpoint checkpoints/{SFT_RUN}/final.pt --run-name {DPO_RUN}"
    if SMOKE:
        cmd += " --n-test 40"
    print(cmd, flush=True)
    get_ipython().system(cmd)
else:
    print("RUN_DPO is False -- skipping")

## 6. Evaluation: base → SFT → DPO

Writes to `EVAL_OUT` (`results/eval`, or `results/eval_oasst1` with `CHAIN_DATASET`).

In [ ]:
if RUN_EVAL:
    cmd = ("python scripts/evaluate.py --base checkpoints/gqa_base/final.pt"
           f" --sft checkpoints/{SFT_RUN}/final.pt --dpo checkpoints/{DPO_RUN}/final.pt"
           f" --out {EVAL_OUT}")
    if SMOKE:
        cmd += " --n-test 40 --n-ppl-stories 100"
    print(cmd, flush=True)
    get_ipython().system(cmd)
    from IPython.display import Markdown, display
    display(Markdown(Path(EVAL_OUT, "samples.md").read_text()))
else:
    print("RUN_EVAL is False -- skipping")

## 7. Benchmark re-run (after the 6 Nov SDPA `enable_gqa` fix)

Random weights, no checkpoints needed, ~15 min. Writes `results/benchmark_enable_gqa/benchmark.json`;
`results/benchmark/` (1 Nov, before the fix) is kept for the before/after comparison.

In [ ]:
if RUN_BENCH:
    get_ipython().system("python scripts/benchmark_attention.py --out results/benchmark_enable_gqa")
else:
    print("RUN_BENCH is False -- skipping")

## 8. Package results for GitHub

Checkpoints stay on Kaggle (too big for GitHub). The small files this run
**created or changed** (per `git status`, so committed results that were only
cloned are left out) go in `track_d_results.zip`: download it from the
**Output** tab, unzip into the repo folder on your laptop, check `git status`, commit.

In [ ]:
import zipfile
# new or modified files only; fall back to everything if git is unavailable
st = subprocess.run(["git", "status", "--porcelain", "--untracked-files=all", "--",
                     "results", "data/preference"], capture_output=True, text=True)
if st.returncode == 0:
    keep = [Path(line[3:].strip().strip('"')) for line in st.stdout.splitlines() if line.strip()]
    keep = [p for p in keep if p.is_file()]
else:
    keep = [p for p in Path("results").rglob("*") if p.is_file()]
    keep += [p for p in Path("data/preference").glob("*") if p.is_file()]
with zipfile.ZipFile("/kaggle/working/track_d_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, p)
        print(f"{p}  {p.stat().st_size / 1e3:.0f} kB")
print("\n-> /kaggle/working/track_d_results.zip")